# Selenium to Playwright Code Converter

This notebook converts Selenium Java code to Microsoft Playwright Java code using the Groq LLM API.

## Steps
1. Install required Python packages
2. Provide your Groq API key
3. Paste your Selenium code
4. Run the converter to get Playwright code
5. Download the generated `.java` file

## Step 1: Install Required Packages

In [ ]:
!pip install requests

## Step 2: Configuration

Set your Groq API key and the Selenium code you want to convert.

In [ ]:
import requests
import json
import re
import os

# -------------------------------------------------------
# Set your Groq API key here (or use an environment variable)
# -------------------------------------------------------
API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")

LLM_API_URL = "https://api.groq.com/openai/v1/chat/completions"

# -------------------------------------------------------
# Paste your Selenium Java code below
# -------------------------------------------------------
selenium_code = """
WebDriver driver = new ChromeDriver();
driver.get("https://login.salesforce.com/?locale=in");
Wait<WebDriver> wait = new WebDriverWait(driver, Duration.ofSeconds(2));
WebElement usernameInput = wait.until(ExpectedConditions.elementToBeClickable(By.xpath("//input[@name='username']")));
usernameInput.sendKeys("your_username");
WebElement passwordInput = wait.until(ExpectedConditions.elementToBeClickable(By.xpath("//input[@type='password']")));
passwordInput.sendKeys("your_password");
WebElement rememberMeCheckbox = wait.until(ExpectedConditions.elementToBeClickable(By.xpath("//input[@type='checkbox']")));
rememberMeCheckbox.click();
WebElement loginButton = wait.until(ExpectedConditions.elementToBeClickable(By.xpath("//input[@type='submit']")));
loginButton.click();
"""

print("Configuration loaded. API URL:", LLM_API_URL)

## Step 3: Call the LLM API to Convert Selenium Code to Playwright

In [ ]:
def generate_playwright_code(selenium_code: str) -> str:
    """
    Sends the Selenium code to the Groq LLM API and returns the raw LLM response.
    """
    if not selenium_code or not selenium_code.strip():
        return "No valid selenium code details to convert to playwright"

    system_message = (
        "You are a helpful assistant that converts Selenium code to Microsoft Playwright code with latest playwright version. "
        "- Launch browser in non-headless mode"
        "- Your response must contain only Playwright code enclosed in a single code block using triple backticks (```java ... ```). "
        "- Use only standard and correct imports (e.g.,import com.microsoft.playwright.*, import com.microsoft.playwright.BrowserType.LaunchOptions;). "
        "- Add package as automation.tests"
        "- Write comments on the code"
        "- take screenshot before quitting browser, create png file, store image and import file path from java"
        "- Quit Browser"
    )

    user_prompt = f"Convert selenium code {selenium_code} to playwright code using latest microsoft playwright version"

    payload = {
        "model": "llama-3.3-70b-specdec",
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user",   "content": user_prompt}
        ],
        "temperature": 0.02,
        "max_tokens": 1000,
        "top_p": 0.1
    }

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    }

    response = requests.post(LLM_API_URL, headers=headers, data=json.dumps(payload))
    response.raise_for_status()
    return response.text


print("Calling LLM API...")
llm_raw_output = generate_playwright_code(selenium_code)
print("LLM API call complete.")

## Step 4: Extract and Enrich the Generated Java Code

In [ ]:
REQUIRED_IMPORTS = """import com.microsoft.playwright.*;
import com.microsoft.playwright.BrowserType.LaunchOptions;
import com.microsoft.playwright.options.LoadState;
import com.microsoft.playwright.Page;
import com.microsoft.playwright.options.WaitForSelectorState;
import com.microsoft.playwright.Browser;
"""


def add_missing_imports(java_code: str) -> str:
    """Ensures all required Playwright imports are included in the generated code."""
    if "import com.microsoft.playwright.*" not in java_code:
        java_code = REQUIRED_IMPORTS + "\n" + java_code
    return java_code


def extract_java_code(llm_response: str) -> str:
    """
    Parses the raw LLM API JSON response and extracts the Java code block.
    """
    try:
        data = json.loads(llm_response)
        content = data["choices"][0]["message"]["content"].strip()

        # Extract code from ```java ... ``` block
        if "```java" in content:
            start = content.index("```java")
            end = content.rindex("```")
            if end > start:
                content = content[start + len("```java"):end].strip()
        elif "package" in content:
            idx = content.index("package")
            content = content[idx:].strip()

        return add_missing_imports(content)
    except Exception as e:
        print(f"Error parsing LLM response: {e}")
        return add_missing_imports(llm_response)


def extract_class_name(java_code: str) -> str:
    """Extracts the public class name from the Java code."""
    match = re.search(r"public\s+class\s+(\w+)", java_code)
    return match.group(1) if match else None


# Extract the Java code
final_test_code = extract_java_code(llm_raw_output)

# Determine the output file name
class_name = extract_class_name(final_test_code)
output_file_name = f"{class_name}.java" if class_name else "GeneratedPlaywrightTest.java"

print(f"Detected class name : {class_name}")
print(f"Output file name    : {output_file_name}")
print()
print("=" * 60)
print("Generated Playwright Code:")
print("=" * 60)
print(final_test_code)

## Step 5: Save the Generated Code to a File

In [ ]:
with open(output_file_name, "w") as f:
    f.write(final_test_code)

print(f"Generated test code written to '{output_file_name}'")

## Step 6: Download the Generated File

Run the cell below to download the generated Java file to your local machine.

In [ ]:
from google.colab import files
files.download(output_file_name)
print(f"Download initiated for '{output_file_name}'")